# Comparing Weight-based and Weightless Neural Networks for Handwritten Digit Recognition

In [1]:
!pip install torch torchvision numpy

In [2]:
# 1. Uninstall any broken pybind11 versions just in case
!pip uninstall -y pybind11 wisardpkg

# 2. Force the installation of an older, compatible pybind11 (v2.x)
!pip install "pybind11<3.0"

# 3. Install wisardpkg without using the cache
!pip install --no-cache-dir wisardpkg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.3/243.3 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.5/126.5 kB 5.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for wisardpkg: filename=wisardpkg-1.6.3-cp312-cp312-linux_x86_64.whl size=3331015 sha256=3a4922fdb3fb46151ee81bf078c0fb145587de38f9a961e3e798634e45d40be9
  Stored in directory: /tmp/pip-ephem-wheel-cache-hdb9_852/wheels/ac/a2/68/6af3d7fbcb1c34042a3a59650c77bec73ab7dcc1048750e464
Successfully built wisardpkg


- In standard model, normalize data to mean and standard deviation
- In WiSARD, binarize it
    - Thresholding: anything above 0.5 = 1, anything below 0.5 = 0

In [11]:
import torch
from torchvision import datasets, transforms
import wisardpkg as wnn
import numpy as np

In [8]:
# 1. Load the MNIST dataset w. Torch
transform = transforms.Compose([
    transforms.ToTensor(),
    # WiSARD needs 1D binary vectors, so flatten 28x288 to 784
    transforms.Lambda(lambda x: x.view(-1)),
])

train_set = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_set = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# 2. Binarize data w. threshold 0.5
# WiSARD requires lists of ints or np arrays
# Using 127 because [0, 255] and 127 = 0.5 * 255
X_train = (train_set.data.view(-1, 784).numpy() > 127).astype(int).tolist()
y_train = [str(label) for label in train_set.targets.numpy()]

X_test = (test_set.data.view(-1, 784).numpy() > 127).astype(int).tolist()
y_test = test_set.targets.numpy()

100%|██████████| 9.91M/9.91M [00:00<00:00, 64.7MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.67MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 14.7MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.77MB/s]


In [10]:
# Implement WiSARD

# Define parameters
addressSize = 10 # How many bits / RAM neuron
bleaching = True # Handles ties bw classes

# Initialize model
wnn_wisard = wnn.Wisard(addressSize, bleachingActivated=bleaching)

# Training - one pass, no backprop
print("Training WiSARD...")
wnn_wisard.train(X_train, y_train)

# Classification / testing
print("Testing WiSARD...")
precictions = wnn_wisard.classify(X_test)

Training WiSARD...
Testing WiSARD...


In [12]:
# wisardpkg returns string labels, so convert to int
y_pred = np.array([int(p) for p in precictions])

# Accuracy
accuracy = np.mean(y_pred == y_test)
print(f"WiSARD Accuracy: {accuracy * 100:.2f}%")

WiSARD Accuracy: 78.82%
